In [1]:
# %% [markdown]
# # Notebook 5 : Prédiction de l'Indice de Risque Vectoriel (VRI)
# 
# ## Contexte
# Le VRI évalue le risque de prolifération des moustiques vecteurs (paludisme, dengue) à partir de conditions météo :
# - Température optimale autour de 25°C
# - Humidité relative optimale autour de 70%
# - Présence d'eau (précipitations)
# - Saison sèche (facteur défavorable)
# 
# ## Objectif
# Construire un modèle supervisé pour prédire le VRI à partir des variables météo disponibles.

# %% [markdown]
# ### 1. Import des bibliothèques

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# XGBoost
import xgboost as xgb

# Interprétabilité
import shap

# Sauvegarde
import joblib

from pathlib import Path

DATASET_PATH = Path("C:/Users/LENOVO/Desktop/hackathon/HACKATHON-INDABAX-CAMEROON-2026-main/data/Dataset_complet_Meteo.csv")



# %% [markdown]
# ### 2. Chargement et exploration initiale

# Chargement (adapter le chemin)
df = pd.read_csv(DATASET_PATH, parse_dates=['time'])
print("Shape:", df.shape)
print("\nColonnes disponibles:")
print(df.columns.tolist())

# Aperçu
df.head()

# %% [markdown]
# **Vérification des variables nécessaires au calcul du VRI :**
# - temperature_2m_mean
# - relative_humidity_2m_mean
# - precipitation_sum
# - precipitation_hours (si absent, on l'estime)
# - is_dry_season (à construire)

# Vérifier la présence de precipitation_hours
if 'precipitation_hours' not in df.columns:
    print("⚠️ precipitation_hours manquant. Création à partir de precipitation_sum (seuil > 0.1 mm/h supposé sur 24h)")
    # Approximation : nombre d'heures de pluie = precipitation_sum / 0.1  (si intensité moyenne 0.1 mm/h)
    df['precipitation_hours'] = np.clip(df['precipitation_sum'] / 0.1, 0, 24)
    # Alternative plus réaliste : on pourrait utiliser un modèle, mais pour l'exemple on garde cette heuristique

# %% [markdown]
# ### 3. Analyse exploratoire ciblée VRI

# Distribution des variables clés
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df['temperature_2m_mean'], kde=True, ax=axes[0,0])
axes[0,0].set_title('Température moyenne (°C)')
sns.histplot(df['relative_humidity_2m_mean'], kde=True, ax=axes[0,1])
axes[0,1].set_title('Humidité relative moyenne (%)')
sns.histplot(df['precipitation_sum'], kde=True, ax=axes[1,0])
axes[1,0].set_title('Précipitations (mm)')
sns.histplot(df['precipitation_hours'], kde=True, ax=axes[1,1])
axes[1,1].set_title('Heures de pluie')
plt.tight_layout()
plt.show()

# Analyse par région et saison
df['month'] = pd.to_datetime(df['time']).dt.month
df['is_dry_season'] = df['month'].isin([11,12,1,2,3]).astype(int)
# Visualisation de la température moyenne par saison et région
plt.figure(figsize=(12,5))
sns.boxplot(data=df, x='region', y='temperature_2m_mean', hue='is_dry_season')
plt.xticks(rotation=45)
plt.title('Température par région et saison (0=pluvieuse, 1=sèche)')
plt.show()

# %% [markdown]
# ### 4. Feature engineering : construction de la cible VRI

# Définition des fonctions gaussiennes pour les plages optimales
def gaussian_opt(x, x0, sigma):
    """Fonction gaussienne centrée sur x0 avec écart-type sigma, normalisée à 1 au centre."""
    return np.exp(-((x - x0)**2) / (2 * sigma**2))

# Calcul du VRI selon la formule du document
def compute_vri(row):
    T = row['temperature_2m_mean']
    RH = row['relative_humidity_2m_mean']
    P = row['precipitation_sum']
    H = row['precipitation_hours']
    S = row['is_dry_season']   # 1 si saison sèche, 0 sinon
    
    # Composantes optimales
    T_opt = gaussian_opt(T, x0=25, sigma=5)
    RH_opt = gaussian_opt(RH, x0=70, sigma=15)
    # Facteur eau : présence d'eau (pluie ou heures de pluie)
    # On utilise une fonction sigmoïde pour modéliser l'effet seuil
    water_factor = 1 - np.exp(-P / 10)  # croît avec la pluie, saturé vers 1
    # Alternative : utiliser precipitation_hours normalisé
    # water_factor = np.clip(H / 12, 0, 1)  # max à 12h de pluie
    
    # Saison sèche réduit le risque (facteur (1-S))
    vri = T_opt * RH_opt * water_factor * (1 - S)
    return vri

# Application
df['VRI'] = df.apply(compute_vri, axis=1)

# Vérification des valeurs
print("VRI - min:", df['VRI'].min(), "max:", df['VRI'].max(), "moyenne:", df['VRI'].mean())

# Visualisation de la distribution du VRI
plt.figure(figsize=(10,4))
sns.histplot(df['VRI'], bins=50, kde=True)
plt.title('Distribution de l\'Indice de Risque Vectoriel (VRI)')
plt.xlabel('VRI')
plt.show()

# %% [markdown]
# ### 5. Features explicatives pour la prédiction du VRI

# On ne doit pas utiliser les variables qui sont déjà des transformations directes de la cible.
# On choisit des features météo brutes et des features temporelles.

# Features temporelles
df['year'] = pd.to_datetime(df['time']).dt.year
df['day_of_year'] = pd.to_datetime(df['time']).dt.dayofyear
df['sin_doy'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['cos_doy'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

# Variables glissantes (rolling) pour capturer l'historique récent
# (évite la fuite de données : on utilise les valeurs passées)
for lag in [1, 2, 3]:
    df[f'temp_lag_{lag}'] = df.groupby('city')['temperature_2m_mean'].shift(lag)
    df[f'hum_lag_{lag}'] = df.groupby('city')['relative_humidity_2m_mean'].shift(lag)
    df[f'precip_lag_{lag}'] = df.groupby('city')['precipitation_sum'].shift(lag)

# Moyennes mobiles sur 3 et 7 jours
df['temp_roll3'] = df.groupby('city')['temperature_2m_mean'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['hum_roll3'] = df.groupby('city')['relative_humidity_2m_mean'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['precip_roll7'] = df.groupby('city')['precipitation_sum'].transform(lambda x: x.rolling(7, min_periods=1).sum())

# Sélection des features (exclure les colonnes qui ne sont pas disponibles à l'inférence)
feature_cols = [
    'temperature_2m_mean', 'relative_humidity_2m_mean', 'precipitation_sum',
    'precipitation_hours', 'wind_speed_10m_max', 'shortwave_radiation_sum',
    'sin_doy', 'cos_doy',
    'temp_lag_1', 'temp_lag_2', 'temp_lag_3',
    'hum_lag_1', 'hum_lag_2', 'hum_lag_3',
    'precip_lag_1', 'precip_lag_2', 'precip_lag_3',
    'temp_roll3', 'hum_roll3', 'precip_roll7'
]

# Supprimer les lignes avec des NaN créés par les lags
df_model = df.dropna(subset=feature_cols + ['VRI']).copy()

X = df_model[feature_cols]
y = df_model['VRI']

print("Taille après suppression des NaN:", X.shape)

# %% [markdown]
# ### 6. Split temporel (pas aléatoire)

# On ordonne par date
df_model = df_model.sort_index()  # l'index est la date
# Split : avant 2024 pour l'entraînement, à partir de 2024 pour le test
train_mask = df_model.index.year < 2024
test_mask = df_model.index.year >= 2024

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train: {X_train.shape[0]} observations")
print(f"Test: {X_test.shape[0]} observations")

# Normalisation (important pour les modèles linéaires et XGBoost)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# %% [markdown]
# ### 7. Entraînement et comparaison de modèles

models = {
    'Ridge': Ridge(alpha=1.0),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    print(f"{name:15} | MAE: {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f}")

# Meilleur modèle basé sur le MAE
best_model_name = min(results, key=lambda x: results[x]['MAE'])
best_model = models[best_model_name]
print(f"\n✅ Meilleur modèle : {best_model_name}")

# %% [markdown]
# ### 8. Optimisation hyperparamètres (optionnelle)

# Exemple pour RandomForest
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}
# Attention : GridSearchCV avec TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=3)
grid_rf = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, 
                       cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_rf.fit(X_train_scaled, y_train)
best_rf = grid_rf.best_estimator_
print("Meilleur RandomForest:", grid_rf.best_params_)
# Comparer avec le meilleur précédent
y_pred_rf = best_rf.predict(X_test_scaled)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
print(f"MAE après optimisation: {mae_rf:.4f}")
if mae_rf < results[best_model_name]['MAE']:
    best_model = best_rf
    best_model_name = 'RandomForest_optimized'
    print("Nouveau meilleur modèle !")

# %% [markdown]
# ### 9. Sauvegarde du meilleur modèle et du scaler

joblib.dump(best_model, 'best_vri_model.pkl')
joblib.dump(scaler, 'scaler_vri.pkl')
# Sauvegarder aussi la liste des features
with open('vri_features.txt', 'w') as f:
    for feat in feature_cols:
        f.write(feat + '\n')
print("Modèle et scaler sauvegardés.")

# %% [markdown]
# ### 10. Interprétabilité avec SHAP

# Pour les modèles arborescents (RandomForest, XGBoost, GradientBoosting)
if best_model_name in ['RandomForest', 'RandomForest_optimized', 'GradientBoosting', 'XGBoost']:
    # Utiliser un sous-ensemble de test pour accélérer
    X_sample = X_test_scaled[:200]
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_sample)
    
    # Summary plot global
    plt.figure(figsize=(10,6))
    shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, show=False)
    plt.title("Importance SHAP des features pour le VRI")
    plt.tight_layout()
    plt.savefig('vri_shap_summary.png')
    plt.show()
    
    # Force plot pour une prédiction individuelle (exemple première observation du test)
    shap.initjs()
    force_plot = shap.force_plot(explainer.expected_value, shap_values[0,:], 
                                 X_sample[0,:], feature_names=feature_cols, matplotlib=True)
    plt.savefig('vri_shap_force.png')
    plt.show()
else:
    # Pour Ridge, utiliser les coefficients
    coefficients = pd.Series(best_model.coef_, index=feature_cols)
    coefficients.sort_values().plot(kind='barh', figsize=(10,6))
    plt.title('Coefficients du modèle Ridge (importance linéaire)')
    plt.tight_layout()
    plt.savefig('vri_ridge_coef.png')
    plt.show()

# %% [markdown]
# ### 11. Exemple d'inférence sur de nouvelles données

def predict_vri(new_data_df):
    """
    new_data_df : DataFrame avec les mêmes colonnes que feature_cols (sans VRI)
    """
    # Vérifier les colonnes
    missing = set(feature_cols) - set(new_data_df.columns)
    if missing:
        raise ValueError(f"Colonnes manquantes : {missing}")
    # Normaliser
    X_new = new_data_df[feature_cols].values
    X_new_scaled = scaler.transform(X_new)
    # Prédiction
    preds = best_model.predict(X_new_scaled)
    return preds

# Exemple : prendre les 5 premières lignes du test
sample = X_test.iloc[:5]
preds = predict_vri(sample)
print("Prédictions VRI pour 5 échantillons :", preds)

# %% [markdown]
# ### 12. Conclusion
# 
# - Le VRI a été calculé selon les spécifications du document.
# - Le meilleur modèle obtenu est `{best_model_name}` avec un MAE de `{results[best_model_name]['MAE']:.4f}`.
# - Les features les plus importantes (selon SHAP) sont : ... (à visualiser).
# - Le modèle peut être utilisé pour estimer le risque vectoriel à partir des prévisions météo.

# Affichage récapitulatif
print("\n=== RÉCAPITULATIF ===")
print(f"Modèle final : {best_model_name}")
print(f"MAE sur test : {mean_absolute_error(y_test, best_model.predict(X_test_scaled)):.4f}")
print(f"RMSE sur test : {np.sqrt(mean_squared_error(y_test, best_model.predict(X_test_scaled))):.4f}")
print(f"R² sur test : {r2_score(y_test, best_model.predict(X_test_scaled)):.4f}")

ImportError: DLL load failed while importing _multiarray_umath: Le module spécifié est introuvable.

ImportError: DLL load failed while importing _multiarray_umath: Le module spécifié est introuvable.

ImportError: DLL load failed while importing _multiarray_umath: Le module spécifié est introuvable.

ImportError: DLL load failed while importing _multiarray_umath: Le module spécifié est introuvable.

ImportError: DLL load failed while importing _multiarray_umath: Le module spécifié est introuvable.

ImportError: DLL load failed while importing _multiarray_umath: Le module spécifié est introuvable.

ImportError: DLL load failed while importing _multiarray_umath: Le module spécifié est introuvable.

ImportError: DLL load failed while importing _multiarray_umath: Le module spécifié est introuvable.

ImportError: numpy._core.multiarray failed to import

Points clés à adapter selon votre dataset réel
precipitation_hours : si la colonne n’existe pas, l’heuristique utilisée (precipitation_sum / 0.1) est approximative. Vous pouvez aussi ignorer cette variable et utiliser seulement precipitation_sum dans le water_factor.

Gestion des valeurs manquantes : avant le calcul des lags, il faut éventuellement interpoler les séries temporelles par ville. Ajoutez :

python
df = df.groupby('city').apply(lambda g: g.interpolate(method='linear'))



Choix des features : n’incluez pas is_dry_season comme feature car elle est directement utilisée dans la construction du VRI (cela créerait une fuite). Nous avons préféré utiliser des variables cycliques (sin/cos) pour capturer la saisonnalité.

Split temporel : les données vont jusqu’en décembre 2025. Utilisez par exemple 2020-2023 pour l’entraînement, 2024-2025 pour le test.

Interprétabilité : SHAP fonctionne bien pour les modèles arborescents. Si vous utilisez un réseau de neurones, utilisez shap.DeepExplainer.

Structure des fichiers générés
best_vri_model.pkl : modèle entraîné

scaler_vri.pkl : standardiseur

vri_features.txt : liste des features utilisées

vri_shap_summary.png : graphique d’importance SHAP

vri_ridge_coef.png (si Ridge meilleur)